# Reviewer-Response Agent — Live Demo

This notebook walks through every stage of the pipeline against the sample manuscript + reviews in `data/`. Use it to:

- Verify the install works (`uv sync` first).
- Inspect intermediate outputs (chunks, embeddings, parsed comments, per-comment trace).
- Render the final rebuttal letter inline.

Make sure `.env` exists with `GEMINI_API_KEY` set (or `LLM_PROVIDER=ollama` if you've pointed it at your home server).

## 1. Setup — config + provider

Loads `.env`, picks the LLM backend, and verifies the API key is reachable.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

from response_agent.config import load_config
from response_agent.llm import get_provider

cfg = load_config()
print(f"Provider:       {cfg.provider}")
if cfg.provider == "gemini":
    print(f"Chat model:     {cfg.gemini_chat_model}")
    print(f"Embed model:    {cfg.gemini_embed_model}")
    print(f"Min interval:   {cfg.gemini_min_interval}s   max retries: {cfg.gemini_max_retries}")
else:
    print(f"Ollama host:    {cfg.ollama_host}")
    print(f"Chat model:     {cfg.ollama_chat_model}")
    print(f"Embed model:    {cfg.ollama_embed_model}")

llm = get_provider(cfg)
print("LLM client ready ✓")

## 2. Load inputs

In [ ]:
from response_agent.ingest import load_manuscript, load_reviews

DATA = Path("data")
manuscript = load_manuscript(DATA / "manuscript.pdf")
reviews    = load_reviews(DATA / "reviews.txt")

print(f"Manuscript: {len(manuscript):,} chars")
print(f"Reviews:    {len(reviews):,} chars")
print("\n--- first 300 chars of manuscript ---")
print(manuscript[:300], "…")

## 3. Chunk + embed the manuscript

Builds the in-memory ChunkIndex used by the retrieval layer.

In [ ]:
from response_agent.chunking import chunk_text
from response_agent.retrieval import build_index

chunks = chunk_text(manuscript)
print(f"{len(chunks)} chunks (chunk_chars=1800, overlap=200)")

index = build_index(chunks, llm)
print(f"Index: {index.embeddings.shape[0]} vectors × {index.embeddings.shape[1]} dims")

## 4. PARSE — split raw reviews into atomic comments

In [ ]:
from response_agent.agents import parse_reviews

comments = parse_reviews(llm, reviews)
print(f"Parsed {len(comments)} atomic comments.\n")
for i, c in enumerate(comments, 1):
    snippet = c['comment'][:120] + ('…' if len(c['comment']) > 120 else '')
    print(f"  [{i:>2}] {c['reviewer']}: {snippet}")

## 5. Step through ONE comment end-to-end

Verbose walkthrough of the agent loop on the first comment, so you can see what the Draft → Critique → Verify → Refine path looks like in isolation.

In [ ]:
from response_agent.agents import critique, draft_response, adversarial_followup
from response_agent.retrieval import retrieve
from response_agent.verifier import verify_citations

demo_comment = comments[0]
print("REVIEWER:", demo_comment['reviewer'])
print("COMMENT:", demo_comment['comment'])
print()

ctx_chunks = retrieve(index, demo_comment['comment'], llm, k=4)
print(f"Retrieved {len(ctx_chunks)} chunks. First chunk preview:\n")
print(ctx_chunks[0][:300], "…\n")

context = "\n\n---\n\n".join(ctx_chunks)
draft = draft_response(llm, demo_comment['comment'], context)
print("--- DRAFT ---\n", draft, "\n")

verdict = critique(llm, demo_comment['comment'], draft, context)
cite_issues = verify_citations(draft, manuscript)
print("--- CRITIC ---")
print("ok:    ", verdict.get('ok'))
print("issues:", verdict.get('issues') or '—')
print("verifier:", cite_issues or '(no citation issues)')
print()

if not verdict.get('ok') or cite_issues:
    merged = " ".join(filter(None, [verdict.get('issues', ''), cite_issues]))
    refined = draft_response(llm, demo_comment['comment'], context, refine_note=merged)
    print("--- REFINED ---\n", refined)
else:
    print("(no refinement needed)")

## 6. Run the full pipeline

Same as `uv run python main.py`. Writes `outputs/rebuttal.md` and `outputs/rebuttal.pdf`.

In [ ]:
from response_agent.pipeline import run

out = run(
    manuscript_path=DATA / "manuscript.pdf",
    reviews_path=DATA / "reviews.txt",
    output_path=Path("outputs/rebuttal.md"),
    top_k=4,
    max_refine_passes=1,
    adversarial=False,   # flip to True for the diagnostic follow-ups (doubles LLM calls)
)
print("\nWritten:", out, "and", out.with_suffix('.pdf'))

## 7. Render the final rebuttal letter inline

In [ ]:
letter = Path("outputs/rebuttal.md").read_text()
print(f"{len(letter):,} chars\n")
display(Markdown(letter))

## 8. (Optional) Adversarial follow-ups

Generates a hostile-reviewer follow-up per comment. Diagnostic only — these are *not* in the rebuttal letter. Useful before submitting to anticipate what round 2 will look like.

In [ ]:
# Re-run with the adversarial agent enabled, but only on the first 3 comments to keep it fast.
for i, c in enumerate(comments[:3], 1):
    ctx = "\n\n---\n\n".join(retrieve(index, c['comment'], llm, k=4))
    draft = draft_response(llm, c['comment'], ctx)
    followup = adversarial_followup(llm, c['comment'], draft, ctx)
    print(f"\n[{i}] {c['reviewer']}")
    print("  comment :", c['comment'][:120], "…")
    print("  follow-up:", followup or '(none — response is airtight)')

## Done

- Modular pipeline drove the same prompts as `main.py` and `app.py`.
- Outputs: `outputs/rebuttal.md` + `outputs/rebuttal.pdf`.
- To swap to a local Ollama server: set `LLM_PROVIDER=ollama` in `.env` and restart this kernel — every cell above will then call your Ollama instead, with no code change.